<!-- DATA PROVIDER INSTRUCTIONS

1. Provide the name of your dataset, replacing the bracketed placeholder text.
2. Update the Registry of Open Data landing page URL, by replacing the bracketed placeholder text. The [REGISTRY_YAML_NAME] will correspond to the name of the YAML document in your pull request to the Registry of Open Data on Github, minus the .yaml file extension.
3. Remove these comment blocks when you have completed each section.

DATA PROVIDER INSTRUCTIONS -->

# Get to Know a Dataset: BossDB

This notebook serves as a guided tour of the [Brain Observatory Storage Service & Database (BossDB)](https://registry.opendata.aws/bossdb/), focusing on the MICrONS dataset. More usage examples, tutorials, and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).

<!-- DATA PROVIDER INSTRUCTIONS

The goal of this section is to orient users to the structure of your dataset. 

1. How are key prefixes and objects organized in your S3 bucket?
2. What kinds of filetypes are represented in your dataset?
3. Explain with text what users are expected to encounter, and then demonstrate with code the organizational framework you applied when creating your dataset.
4. The responses to each question section are meant to be expanded or replaced as dictated by your dataset

DATA PROVIDER INSTRUCTIONS -->

### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.

[BossDB](https://bossdb.org) is a volumetric database for 3D and 4D neuroscience data, focused on Electron Microscopy and X-ray Micro/Nano Tomography. Datasets are stored in an AWS Open Data bucket and follow a consistent, hierarchical prefix structure that mirrors the [BossDB data model](https://metadata.bossdb.org/DataModel). This structure is designed to support large-scale, multi-resolution neuroscience imaging data across a variety of datasets.

At a high level, datasets in the BossDB S3 Open Data bucket are organized as:
```
s3://bossdb-open-data/<project>/<collection>/<experiment>/<channel>/
```

Where:
1. ```Project```: Represents a top-level project name, typically the data owner or consortia and the year (e.g. smith2026)
2. ```Collection```: Groups related samples or acquisitions within a project, often corresponding to a specimen, subject, or study subset
3. ```Experiment```: Represents a specific imaging experiment or modality applied to a collection
4. ```Channel```: Encodes a single data stream, such as raw image data
 
For this tutorial, we focus on the Minnie subset of the MICrONS dataset. Documentation for this dataset can be found at:
1. BossDB MICrONS Minnie project page: https://bossdb.org/project/microns_minnie2021
2. BossDB S3 prefix: s3://bossdb-open-data/iarpa_microns/minnie 

First we will import the Python libraries required throughout this notebook.

In [ ]:
# This notebook requires the following additional libraries
!pip install boto3 requests matplotlib pillow intern

# Import the libraries required for this notebook
import json
import matplotlib.pyplot as plt
import boto3, matplotlib.pyplot as plt, requests

# Installed libraries
from PIL import Image
from botocore import UNSIGNED
from botocore.config import Config
from pprint import pprint
from intern import array
from PIL import Image
from io import BytesIO

Next, we will define the location of our dataset, create our boto3 S3 client, and list the top level prefixes in our S3 bucket. Here we see there is only one top-level prefix in BossDB's Open Data bucket.

In [ ]:
# Location of the S3 bucket for this dataset
bucket = "bossdb-open-data"

# List the top level of the bucket using boto3. Because this is a public bucket, we don't need to sign requests.
# Here we set the signature version to unsigned, which is required for public buckets.
s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))

# Print the items in the top-level prefixes to see all of the different BossDB project datasets
for item in s3.list_objects_v2(Bucket=bucket, Delimiter='/')['CommonPrefixes']:
    print(item['Prefix'])

At the top level of the BossDB Open Data bucket, prefixes correspond to projects hosted in BossDB. Each project prefix is a dataset that contains one or more collections, which then branch into experiments and channels. This follows the data model structure described in the previous section.

In [ ]:
# List the key prefixes (BossDB data model collections) within the top level of the 'microns' prefix
for item in s3.list_objects_v2(Bucket=bucket, Prefix='iarpa_microns/', Delimiter='/', MaxKeys=10)['CommonPrefixes']:
    print(item['Prefix'])

Looking inside the `iarpa_microns/minnie/` path, there are a variety of different files. Some sub-prefixes, such as `functional_coregistration/` and `functional_data/digital_twin_properties/`, contain tabular data, metadata, and documentation intended for direct user access. These files can be downloaded and inspected. Other sub-prefixes (for example, those under `functional_data/field_seg_masks/`) contain large numbers of small objects with coordinate-based names.

In [ ]:
# List the keys within the 'iarpa_microns/minnie/' prefix.
for item in s3.list_objects_v2(Bucket=bucket, Prefix='iarpa_microns/minnie/', MaxKeys=100)['Contents']:
    print(item['Key'])

<!-- DATA PROVIDER INSTRUCTIONS
This section is meant to orient users of your dataset to the formats present in your dataset, particularly if your dataset includes formats that may be unfamiliar to a general data scientist audience. This section should include:

1. Explanation of data format(s) (very common formats can be very briefly described, while less common
   or domain specific formats should include more explanation as well as links to official documentation)
2. Explanation of why the data format was chosen for your dataset
3. Recommendations around software and tooling to work with this data format
4. Explanation of any dataset-specific aspects to your usage of the format
5. Description of AWS services that may be useful to users working with your data
DATA PROVIDER INSTRUCTIONS -->

### Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?

BossDB hosts a variety of neuroscience data types, and data are stored using multiple formats. The specific formats present depend on the project and collection, but commonly include volumetric imagery, annotations, and segmentations.

Volumetric imagery (e.g., electron microscopy or light-sheet microscopy data) is typically acquired as TIFF stacks and then converted into a chunked representation for storage and access. Image data are stored as many small objects with coordinate-based keys. This layout allows tools to retrieve only the portions of a volume needed for a given task, such as visualization or localized analysis, rather than requiring download of an entire dataset.

For the MICrONS datasets specifically, volumetric imagery and derived data stored in the underlying AWS S3 buckets are provided in the Neuroglancer precomputed format. Direct S3 access returns data in this multiscale, chunked representation, which is compatible with BossDB as well as open-source tools such as [Neuroglancer](https://neuroglancer.bossdb.io/#!%7B%22dimensions%22:%7B%22x%22:%5B8e-9%2C%22m%22%5D%2C%22y%22:%5B8e-9%2C%22m%22%5D%2C%22z%22:%5B4e-8%2C%22m%22%5D%7D%2C%22position%22:%5B120172.5625%2C103932.2578125%2C21365.646484375%5D%2C%22crossSectionScale%22:1%2C%22crossSectionDepth%22:-7.67205949975856%2C%22projectionScale%22:262144%2C%22layers%22:%5B%7B%22type%22:%22image%22%2C%22source%22:%22precomputed://s3://bossdb-open-data/iarpa_microns/minnie/minnie65/em%22%2C%22tab%22:%22source%22%2C%22name%22:%22em%22%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%22precomputed://s3://bossdb-open-data/iarpa_microns/minnie/minnie65/seg%22%2C%22tab%22:%22source%22%2C%22segments%22:%5B%5D%2C%22name%22:%22seg%22%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%22precomputed://s3://bossdb-open-data/iarpa_microns/minnie/minnie65/clefts%22%2C%22tab%22:%22source%22%2C%22segments%22:%5B%5D%2C%22name%22:%22clefts%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%22precomputed://s3://bossdb-open-data/iarpa_microns/minnie/minnie65/nuclei%22%2C%22tab%22:%22source%22%2C%22segments%22:%5B%5D%2C%22name%22:%22nuclei%22%2C%22visible%22:false%7D%5D%2C%22selectedLayer%22:%7B%22visible%22:true%2C%22layer%22:%22nuclei%22%7D%2C%22layout%22:%224panel%22%7D) for visualization.

Many BossDB datasets also include annotation and segmentation data aligned to the underlying image volumes. These data may represent labeled structures, masks, or object identifiers, and are stored using formats appropriate to their structure, including chunked volumes, NumPy arrays, or tabular files. Segmentation and annotation data are typically accessed alongside image data using the same tools and coordinate systems, enabling direct comparison and overlay in visualization and analysis workflows.

Because BossDB hosts multiple data types and formats, users typically combine several access methods depending on their goals:
- Visualization tools for exploring volumetric data (e.g., Neuroglancer)
- BossDB APIs or volume-access libraries for programmatic interaction with image and annotation data (e.g., CAVE)

BossDB datasets are hosted in Amazon Web Services Open Data buckets, allowing users to take advantage of cloud-native services for scalable data access, computation, and analysis. For readers interested in a practical walkthrough of these tools and workflows, there are a variety of associated publications on the [BossDB publications website](https://bossdb.org/publications). In particular, the [Using BossDB Tools to Access, Visualize, and Share Volumetric Neuroscience Data](https://currentprotocols.onlinelibrary.wiley.com/doi/full/10.1002/cpz1.70247) paper provides detailed intruction into BossDB workflows. 

<!-- DATA PROVIDER INSTRUCTIONS
The goal of this section is to demonstrate loading a portion of data from your dataset, and reveal something about its structure.
1. Load an object from S3
2. Show the structure of data in the object
DATA PROVIDER INSTRUCTIONS -->

### Q: Can you show us an example of downloading and loading data from your dataset?

As an example, we will load an image cutout from the MICrONS Minnie dataset using the BossDB Python SDK. This can be used to analyze how volumetric image data are accessed and what their structure looks like once loaded into memory, and this can be adjusted to look at different datasets and views.


In [ ]:
# Access the EM channel for the MICrONS Minnie65 dataset
em = array("bossdb://microns/minnie65_8x8x40/em")

# Download a small volumetric cutout (data returned in ZYX order)
cutout = em[19000:19016, 56298:57322, 79190:80214]


Inspecting the type, shape, and data type of the loaded object can also help clarify how volumetric image data are represented in memory.


In [ ]:
# Inspect the structure of the loaded MICrONS Minnie cutout
print("Type:", type(cutout))
print("Shape (Z, Y, X):", cutout.shape)
print("Data type:", cutout.dtype)

Next, we can also examine an individual slice from the volumetric cutout to understand the structure of the data. Each slice corresponds to a single two-dimensional electron microscopy image, where pixel intensities represent grayscale signal values at nanometer-scale resolution.

In [ ]:
# Select the first Z slice
slice_0 = cutout[0]

print("Min intensity:", slice_0.min())
print("Max intensity:", slice_0.max())


<!-- DATA PROVIDER INSTRUCTIONS
The goal here is to visualize some aspect of your dataset in order to help users understand it. In addition to helping users of your dataset understand the dataset, an additional goal is to impress!

Please demonstrate any data preprocessing or reshaping required for your visualization(s).

https://www.reddit.com/r/dataisbeautiful/ for inspiration.
DATA PROVIDER INSTRUCTIONS -->

### Q: A picture is worth a thousand words. Show us a visual (or several!) from your dataset that either illustrates something informative about your dataset, or that you think might excite someone to dig in further.

The following cell visualizes one 2D slice from the EM volume. This process can also repeated to visualize different regions of the dataset by adjusting the spatial coordinates used in the cutout.

In [ ]:
# Visualize a single EM slice
plt.imshow(cutout[0], cmap="gray")
plt.axis("off")
plt.show()


In addition to programmatic access, BossDB datasets hosted on the S3 Open Data bucket can be explored in Neuroglancer through the BossDB web integration. Neuroglancer streams volumetric data directly from S3 without requiring local downloads. You can simply copy and paste the S3 path you wish to analyze: 
![Neuroglancer visualization of MICrONS Minnie](/bossdb_cookbook/NGL_Minnie.png)

<!-- DATA PROVIDER INSTRUCTIONS
This section is less prescriptive / freeform than previous sections. The goal here is to show an opinionated example of answering a question using your data. The scale of your dataset may preclude a full example, and so feel free to limit the scope of this example (e.g. work on a subset of data). Users should be able to replicate your example in this notebook, and get a sense of how they would scale up.

A "toy" example is better than no example.

Ideally, your example would:
1. Transmit some of your domain & dataset experience to the reader, drawing on your own work as much as possible
2. Provide a jumping off point for users to extend your work, and do novel work of their own.

DATA PROVIDER INSTRUCTIONS -->

### Q: What is one question that you have answered using these data? Can you show us how you came to that answer?

One question that has been addressed using the MICrONS Minnie dataset is: How does synapse-level connectivity relate to neuronal function in mouse visual cortex?

As shown in the [MICrONS publication](https://www.nature.com/articles/s41586-025-08790-w), this dataset was among the first to make this question tractable at scale because it combines dense electron microscopy reconstructions with functional imaging from the same tissue. Earlier connectomics datasets typically provided either detailed anatomy or functional measurements, but not both in a way that allowed direct correspondence between structure and activity.

Using the proofread segmentations and synapse graph from the Minnie65 volume, researchers identified neurons and their synaptic partners at nanometer resolution. These anatomical relationships were then linked to functional response properties derived from two-photon calcium imaging during visual stimulation. By comparing connectivity patterns among neurons with similar and dissimilar response profiles, the analysis revealed structured wiring rules that extend beyond simple spatial proximity.

The Minnie dataset is stored in a precomputed, multiscale format, which allows imagery, segmentations, and synapse annotations to be efficiently streamed through BossDB and accessed using open-source tools such as CloudVolume, TensorStore, and Neuroglancer. A simplified version of this workflow can be reproduced by extracting a small spatial cutout from BossDB, selecting a subset of neurons from the segmentation, querying the synapse graph to determine their connections, and joining those results with available functional annotations. Even at this reduced scale, the example demonstrates how the dataset enables direct analysis of structure–function relationships that were not previously possible.

<!-- DATA PROVIDER INSTRUCTIONS
This section is, like the previous one, intended to be freeform / non-prescriptive. The goal here is to provide a challenge to the community to do something novel with your dataset. That can either be novel in terms of the task, or novel in terms of methodological or computational approach.

Another way to consider this section, is as a wishlist. If you were less constrained by time, cost, skill, etc., what would you like to see achieved using these data? 

The challenge should, however, be somewhat realistic. A challenge that assumes e.g. original data collection, is likely to go unanswered.
DATA PROVIDER INSTRUCTIONS -->

### Q: What is one unanswered question that you think could be answered using these data? Do you have any recommendations or advice for someone wanting to answer this question?

One open question that could be explored using these data is: What fine-scale anatomical features and synaptic connectivity patterns are shared across neurons, and which are rare or specialized?

The MICrONS Minnie dataset provides nanometer-scale reconstructions of neurons and their synapses over a large volume of brain tissue. As described in the [MICrONS publication](https://www.nature.com/articles/s41586-025-08790-w), this level of detail makes it possible to examine features such as axonal and dendritic geometry, synapse placement, and local connectivity patterns that are difficult or impossible to study with lower-resolution data.

Because the dataset includes dense, synapse-level connectivity across many cells, it enables analyses that go beyond individual neurons and focus on how anatomical detail varies across populations. Many of these features have not yet been exhaustively cataloged or compared particularly at this scale. 

The dataset’s availability in BossDB supports this type of exploration by enabling efficient access to imagery, segmentations, and synapse data at multiple scales. Analyses developed on the MICrONS Minnie dataset can then be extended to other BossDB-hosted volumes, enabling broader exploration of anatomical structure and connectivity across datasets and imaging modalities.